# logsumexp-cross-entropy — ex1: numerically stable cross-entropy via logsumexp

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `logsumexp-cross-entropy`. Running the final beacon cell reports progress against the `Loss: logsumexp cross-entropy` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Loss: logsumexp cross-entropy` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`logsumexp-cross-entropy`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "logsumexp-cross-entropy"
DD_SUBTOPIC = "Loss: logsumexp cross-entropy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Cross-entropy via logsumexp — quick refresher

The naive cross-entropy formula

```
loss[i] = -log(softmax(logits[i])[target[i]])
        = -log( exp(logits[i, target[i]]) / sum_k exp(logits[i, k]) )
```

is numerically dangerous: `exp(logits)` overflows for logits > ~88 in float32. The stable rewrite uses `logsumexp`:

```
loss[i] = logsumexp(logits[i]) - logits[i, target[i]]
```

where `logsumexp(x) = log(sum_k exp(x_k - max(x))) + max(x)`. The `-max(x)` shift keeps every exp argument ≤ 0, so no overflow.

`torch.logsumexp(logits, dim=-1)` ships this for you. Use it.

Identity check: with `logits = [0, 0, 0]` the naive softmax is `[1/3, 1/3, 1/3]` and the loss for any target is `log(3) ≈ 1.0986`. The logsumexp form gives `log(3) - 0 = 1.0986` — same answer, no overflow.

### Exercise 1 — numerically stable cross-entropy via logsumexp

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the logsumexp identity to compute cross-entropy as `logsumexp(logits) - logits[arange(B), target]`, avoiding the softmax overflow that the naive formulation triggers for large logits.
> Keywords: cross-entropy, logsumexp, numerical-stability, softmax, overflow
> ```

**KCs targeted:** `logsumexp-cross-entropy`, `arange-fancy-index-cross-entropy`

Implement `cross_entropy_logsumexp(logits, target)`. Per-sample, the loss is

```
loss[i] = logsumexp(logits[i]) - logits[i, target[i]]
```

Return the mean loss across the batch — a 0-D `torch.Tensor`.

Inputs:
- `logits`: shape `(B, C)`, float (any magnitude — the function MUST handle logits up to ~10000 without overflow).
- `target`: shape `(B,)`, integer class indices in `[0, C)`.

Output: scalar tensor (`shape == ()`), mean cross-entropy.

**Use `torch.logsumexp(logits, dim=-1)`** for the first term. It internally subtracts the per-row max before exp, so the formula is stable even when logits are huge.

**For the second term**, you need `logits[arange(B), target]` — the advanced-indexing pattern for picking out one entry per row. (A separate drill covers that pattern in isolation; here we use it.)

**Compare with the naive version.** The naive formula `-log(softmax(logits)[target])` is mathematically identical but overflows for `logits[i, k] > ~88` in float32 — `exp(89) > 3e38 > float32_max`. The test cell stresses this case explicitly.

Do NOT call `torch.nn.functional.cross_entropy` — write the formula directly.

In [ ]:
def cross_entropy_logsumexp(logits: Tensor, target: Tensor) -> Tensor:
    """Compute mean cross-entropy via logsumexp(logits) - logits[arange(B), target]."""
    raise NotImplementedError()


def _test_ex1():
    import math
    # --- baseline: uniform logits → log(C) for any target ---
    logits = t.zeros(4, 3)  # uniform → softmax = 1/3 → -log(1/3) = log(3)
    target = t.tensor([0, 1, 2, 0])
    loss = cross_entropy_logsumexp(logits, target)
    assert loss.shape == (), f'expected scalar, got shape {loss.shape}'
    assert abs(loss.item() - math.log(3)) < 1e-5, (
        f'uniform logits → log(C)={math.log(3):.4f}, got {loss.item():.4f}'
    )

    # --- compare against torch.nn.functional.cross_entropy (the witness) ---
    import torch.nn.functional as F
    logits2 = t.tensor([[2.0, 1.0, 0.1], [0.5, -1.0, 3.0]])
    target2 = t.tensor([0, 2])
    loss2 = cross_entropy_logsumexp(logits2, target2)
    ref2 = F.cross_entropy(logits2, target2)
    assert abs(loss2.item() - ref2.item()) < 1e-5, (
        f'mismatch vs F.cross_entropy: ours={loss2.item():.6f}, ref={ref2.item():.6f}'
    )

    # --- THE STABILITY TEST: logits = 1000 must NOT overflow ---
    big_logits = t.tensor([[1000.0, 999.0, 998.0], [0.0, 500.0, 0.0]])
    big_target = t.tensor([0, 1])
    big_loss = cross_entropy_logsumexp(big_logits, big_target)
    assert t.isfinite(big_loss).item(), (
        f'huge logits must not produce inf/nan; got {big_loss.item()} — '
        'are you using logsumexp, or did you call exp(logits) directly?'
    )
    # The expected loss for row 0 (target 0): logsumexp([1000,999,998]) - 1000
    # = log(exp(0) + exp(-1) + exp(-2)) ~ 0.4076.
    # Row 1 (target 1): logsumexp([0,500,0]) - 500 ~ log(2*exp(-500) + 1) ~ 0 -> 0.
    expected_row0 = math.log(1 + math.exp(-1) + math.exp(-2))
    expected = (expected_row0 + 0.0) / 2
    assert abs(big_loss.item() - expected) < 1e-4, (
        f'huge-logit loss wrong: got {big_loss.item()}, expected {expected:.4f}'
    )

    # --- confirm the naive version WOULD overflow at this scale (sanity for the test) ---
    naive_overflowed = not t.isfinite(t.exp(big_logits)).all().item()
    assert naive_overflowed, (
        'sanity: t.exp(big_logits) should overflow at scale 1000 — '
        'if this assertion fails, the test stress case is too weak'
    )

    # --- batch-mean semantics (not sum) ---
    lo = t.zeros(10, 5)
    ta = t.zeros(10, dtype=t.long)
    loss_mean = cross_entropy_logsumexp(lo, ta)
    assert abs(loss_mean.item() - math.log(5)) < 1e-5, (
        f'must return MEAN (not sum); 10 samples × log(5) summed would be {10*math.log(5):.4f}, '
        f'got {loss_mean.item():.4f}'
    )

    # --- target on the correct class with very high logit → near-zero loss ---
    confident = t.tensor([[100.0, 0.0, 0.0]])  # logits favor class 0 strongly
    loss_confident = cross_entropy_logsumexp(confident, t.tensor([0]))
    assert loss_confident.item() < 1e-5, (
        f'confident-correct logits should yield ~0 loss, got {loss_confident.item()}'
    )
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def cross_entropy_logsumexp(logits: Tensor, target: Tensor) -> Tensor:
    # logsumexp over the class axis — numerically stable
    lse = t.logsumexp(logits, dim=-1)                # shape (B,)
    # pick out per-sample target logit via arange-fancy-index
    B = logits.shape[0]
    picked = logits[t.arange(B), target]              # shape (B,)
    # per-sample loss, then batch mean
    per_sample = lse - picked
    return per_sample.mean()
```

**Why logsumexp survives where naive softmax dies.** `torch.logsumexp(x)` computes `log(sum(exp(x - max(x)))) + max(x)`. The max-shift keeps every exp argument ≤ 0, so the largest term is `exp(0) = 1`. The naive `softmax(x) = exp(x) / sum(exp(x))` doesn't shift — and `exp(1000)` is `inf` in any float type.

**The cross-entropy identity.** For per-sample loss:
```
loss = -log(softmax(x)[target])
     = -log(exp(x[target]) / sum_k exp(x[k]))
     = -x[target] + log(sum_k exp(x[k]))
     = logsumexp(x) - x[target]
```
The substitution is exact; the only difference is numerical.

**Mean vs sum.** `F.cross_entropy` defaults to `reduction='mean'`, which is what most training loops want (loss scale doesn't depend on batch size). `sum` is occasionally used when gradient accumulation across mini-batches needs to be unbiased.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()